In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.types import *

In [0]:
lst = dbutils.fs.ls("/Volumes/workspace/olist/volume_olist/")
display(lst)

path,name,size,modificationTime
dbfs:/Volumes/workspace/olist/volume_olist/olist_customers_dataset.csv,olist_customers_dataset.csv,9033957,1774936917000
dbfs:/Volumes/workspace/olist/volume_olist/olist_geolocation_dataset.csv,olist_geolocation_dataset.csv,61273883,1774936926000
dbfs:/Volumes/workspace/olist/volume_olist/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15438671,1774936920000
dbfs:/Volumes/workspace/olist/volume_olist/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5777138,1774936916000
dbfs:/Volumes/workspace/olist/volume_olist/olist_order_reviews_dataset.csv,olist_order_reviews_dataset.csv,14451670,1774936920000
dbfs:/Volumes/workspace/olist/volume_olist/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1774936920000
dbfs:/Volumes/workspace/olist/volume_olist/olist_products_dataset.csv,olist_products_dataset.csv,2379446,1774936914000
dbfs:/Volumes/workspace/olist/volume_olist/olist_sellers_dataset.csv,olist_sellers_dataset.csv,174703,1774936913000
dbfs:/Volumes/workspace/olist/volume_olist/product_category_name_translation.csv,product_category_name_translation.csv,2613,1774936913000


## Bronze Layer

### Load CSV files to PySpark DataFrame

In [0]:
## Customer Data Set 

df_lst = []







### Creating Bronze Layer Schema in Catalog

In [0]:
geo_loc_ds = spark.read.csv("/Volumes/workspace/olist/volume_olist/olist_geolocation_dataset.csv", header=True)

In [0]:
%sql  -- Bronze layer 
CREATE SCHEMA IF NOT EXISTS workspace.bronze
COMMENT 'Schema For Bronze Layer'

### Creating tables And Inserting Data 

#### Customers Dataset

In [0]:
cust_ds = spark.read.csv("/Volumes/workspace/olist/volume_olist/olist_customers_dataset.csv", header=True, inferSchema=True)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.bronze.olist_customers ( 
  customer_id STRING ,
  customer_unique_id STRING , 
  customer_zip_code_prefix INTEGER ,
  customer_city STRING , 
  customer_state STRING 
) USING DELTA 

In [0]:
cust_ds.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.olist_customers")
print(cust_ds.count())

99441


In [0]:
%sql
SELECT COUNT(*)
FROM workspace.bronze.olist_customers

COUNT(*)
99441


#### Geolocation Dataset

In [0]:
geo_loc_ds = spark.read.csv("/Volumes/workspace/olist/volume_olist/olist_geolocation_dataset.csv", header=True)

In [0]:
geo_loc_ds.printSchema()

root
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: string (nullable = true)
 |-- geolocation_lng: string (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)



In [0]:
geo_loc_ds.show(5)

+---------------------------+-------------------+------------------+----------------+-----------------+
|geolocation_zip_code_prefix|    geolocation_lat|   geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+-------------------+------------------+----------------+-----------------+
|                      01037| -23.54562128115268|-46.63929204800168|       sao paulo|               SP|
|                      01046|-23.546081127035535|-46.64482029837157|       sao paulo|               SP|
|                      01046| -23.54612896641469|-46.64295148361138|       sao paulo|               SP|
|                      01041|  -23.5443921648681|-46.63949930627844|       sao paulo|               SP|
|                      01035|-23.541577961711493|-46.64160722329613|       sao paulo|               SP|
+---------------------------+-------------------+------------------+----------------+-----------------+
only showing top 5 rows


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.bronze.olist_geolocation (
    geolocation_zip_code_prefix STRING,
    geolocation_lat STRING,
    geolocation_lng STRING,
    geolocation_city STRING,
    geolocation_state STRING
)

In [0]:
geo_loc_ds.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.olist_geo_loc")
geo_loc_ds.count()

1000163

In [0]:
%sql -- Validation 
SELECT COUNT(*)
FROM workspace.bronze.olist_geo_loc

COUNT(*)
1000163


#### Order Items

In [0]:

order_items_ds = spark.read.csv("/Volumes/workspace/olist/volume_olist/olist_order_items_dataset.csv",header= True )

In [0]:
display(order_items_ds.head(2))

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93


In [0]:
order_items_ds.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- freight_value: string (nullable = true)



In [0]:
%sql 
DROP TABLE workspace.bronze.olist_order_items;

CREATE TABLE IF NOT EXISTS workspace.bronze.olist_order_items (
    order_id STRING,
    order_item_id STRING,
    product_id STRING,
    seller_id STRING,
    shipping_limit_date STRING,
    price STRING,
    freight_value  STRING
) USING DELTA

In [0]:
order_items_ds.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.olist_order_items")
print(order_items_ds.count())

112650


In [0]:
%sql 
SELECT Count(*)
FROM workspace.bronze.olist_order_items
LIMIT 5

Count(*)
112650


#### Olist Order Payments DataSet

In [0]:
order_payments_ds = spark.read.csv("/Volumes/workspace/olist/volume_olist/olist_order_payments_dataset.csv", header= True)

In [0]:
order_payments_ds.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: string (nullable = true)
 |-- payment_value: string (nullable = true)



In [0]:
%sql
DROP TABLE IF EXISTS workspace.bronze.olist_order_payments;

CREATE TABLE IF NOT EXISTS workspace.bronze.olist_order_payments( 
  order_id STRING,
  payment_sequential STRING,
  payment_type STRING,
  payment_installments STRING,
  payment_value STRING 
) USING DELTA 

In [0]:
display(order_payments_ds.head(3))

order_id,payment_sequential,payment_type,payment_installments,payment_value
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


In [0]:
order_payments_ds.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.olist_order_payments")
print(order_payments_ds.count())

103886


In [0]:
%sql  -- Validation
SELECT COUNT(*)
FROM workspace.bronze.olist_order_payments

COUNT(*)
103886


#### Order Reviews

In [0]:
order_reviews_ds = spark.read.csv("/Volumes/workspace/olist/volume_olist/olist_order_reviews_dataset.csv", header=True)

In [0]:
display(order_reviews_ds.limit(2))

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18 00:00:00,2018-01-18 21:46:59
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10 00:00:00,2018-03-11 03:05:13


In [0]:
order_reviews_ds.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)



In [0]:
%sql
DROP TABLE IF EXISTS workspace.bronze.olist_order_reviews;

CREATE TABLE IF NOT EXISTS workspace.bronze.olist_order_reviews (
  review_id STRING,
  order_id STRING,
  review_score STRING,
  review_comment_title STRING,
  review_comment_message STRING,
  review_creation_date STRING,
  review_answer_timestamp STRING 
) USING DELTA 

In [0]:
order_reviews_ds.write.format("DELTA").mode("OVERWRITE").saveAsTable("workspace.bronze.olist_order_reviews")
print(order_reviews_ds.count())

104162


In [0]:
%sql
SELECT COUNT(*)
FROM workspace.bronze.olist_order_reviews


COUNT(*)
104162


#### Orders

In [0]:
orders_ds = spark.read.csv("/Volumes/workspace/olist/volume_olist/olist_orders_dataset.csv", header=True)

In [0]:
display(orders_ds.head(10))

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,null,null,2017-05-09 00:00:00
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00


#### Products

In [0]:
product_ds = spark.read.csv("/Volumes/workspace/olist/volume_olist/olist_products_dataset.csv",header=True)

#### Sellers

In [0]:

sellers_ds = spark.read.csv("/Volumes/workspace/olist/volume_olist/olist_sellers_dataset.csv", header=True)

#### Products Category name 


In [0]:

products_category_name_tr = spark.read.csv("/Volumes/workspace/olist/volume_list/olist_sellers_data.csv", header=True)